# 01 — Data Collection & Environment Setup

**Phase 1** of the Skill Trend Analysis project.

This notebook:
1. Installs and verifies all required libraries
2. Downloads the **Primary** dataset (Global AI & Data Science Job Market 2020–2026) via `kagglehub`
3. Downloads the **Secondary** dataset (1.3M LinkedIn Jobs & Skills 2024) via `kagglehub`
4. Copies all raw files to `data/raw/`
5. Inspects every file: shape, columns, data types, null counts, head
6. Documents schema differences between datasets
7. Extracts and saves a 10k-row LinkedIn sample for validation

---
## 1.1 — Environment Setup

Install all required Python packages. This cell is designed for **Google Colab** or any fresh Python environment.

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly wordcloud streamlit groq scipy kagglehub

In [ ]:
# Verify all imports and print versions
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import plotly
from wordcloud import WordCloud
import scipy
import kagglehub

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("seaborn     ", sns.__version__)
print("plotly      ", plotly.__version__)
print("scipy       ", scipy.__version__)
print("kagglehub   ", kagglehub.__version__)
print("wordcloud    OK")
print("\n✅ All libraries installed and verified.")

In [ ]:
import os
import shutil
from pathlib import Path

# Create project directory structure
for d in ["data/raw", "data/clean", "assets/charts", "dashboards", "src"]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✓ {d}/")
print("\n✅ Directory structure ready.")

---
## 1.2 — Primary Dataset Acquisition

**Dataset:** Global AI & Data Science Job Market 2020–2026 (Kaggle: `mann14/global-ai-and-data-science-job-market-20202026`)

This dataset contains **5 files**:
| File | Description |
|------|-------------|
| `ai_jobs.csv` | 50,000 job postings with metadata (title, company, location, salary, year) |
| `skills_demand.csv` | 224,605 rows — each row is one skill required by one job (links to `ai_jobs` via `job_id`) |
| `country_ai_trends.csv` | Country-level aggregated trends by year |
| `job_title_mapping.csv` | Standardized job title → role category mapping |
| `data_dictionary.csv` | Column descriptions for all files |

In [ ]:
# Download primary dataset via kagglehub
import kagglehub
from pathlib import Path

primary_path = kagglehub.dataset_download("mann14/global-ai-and-data-science-job-market-20202026")
print("Path to dataset files:", primary_path)
print("\nFiles in download:")
for p in sorted(Path(primary_path).rglob("*")):
    if p.is_file():
        print(f"  {p.name:30s}  {p.stat().st_size:>12,} bytes")

In [ ]:
# Copy all primary files to data/raw/
import shutil

raw_dir = Path("data/raw")
for f in Path(primary_path).iterdir():
    if f.is_file():
        dest = raw_dir / f.name
        shutil.copy2(str(f), str(dest))
        print(f"  Copied: {f.name} → data/raw/ ({dest.stat().st_size:,} bytes)")

print("\n✅ Primary dataset files saved to data/raw/")

### 1.2.1 — Inspect `ai_jobs.csv` (Primary Job Metadata)

In [ ]:
df_jobs = pd.read_csv("data/raw/ai_jobs.csv")

print("Shape:", df_jobs.shape)
print("\nColumns:", df_jobs.columns.tolist())
print("\nData types:")
print(df_jobs.dtypes)
print("\nNull counts:")
print(df_jobs.isnull().sum())
print("\nFirst 5 rows:")
df_jobs.head()

**`ai_jobs.csv` — Schema Documentation**

| Column | Dtype | Nulls | Description |
|--------|-------|-------|-------------|
| `job_id` | str | 0 | Unique job identifier (12-char alphanumeric) |
| `job_title` | str | 0 | Original job title (e.g., Data Scientist, MLOps Engineer) |
| `company_type` | str | 0 | Company type: Startup, MNC, or Research Lab |
| `industry` | str | 0 | Industry sector (e.g., Education, Finance, Healthcare) |
| `country` | str | 0 | Country of job posting |
| `city` | str | 0 | City (may be "Remote") |
| `remote_type` | str | 0 | Remote / Hybrid / Onsite |
| `experience_level` | str | 0 | Entry, Mid, or Senior |
| `min_experience_years` | int64 | 0 | Minimum years of experience required |
| `salary_min_usd` | int64 | 0 | Minimum annual salary in USD |
| `salary_max_usd` | int64 | 0 | Maximum annual salary in USD |
| `employment_type` | str | 0 | Full-time, Part-time, or Contract |
| `posted_year` | int64 | 0 | Year the job was posted (2020–2026) |
| `company_size` | str | 0 | Small, Medium, or Large |

> **Note:** This file has NO skills column. Skills are in the separate `skills_demand.csv` file, joined via `job_id`.
> The `job_id` values may have a character-shift mismatch with `skills_demand.job_id` — this is handled during data cleaning.

### 1.2.2 — Inspect `skills_demand.csv` (Skill Requirements)

In [ ]:
df_skills = pd.read_csv("data/raw/skills_demand.csv")

print("Shape:", df_skills.shape)
print("\nColumns:", df_skills.columns.tolist())
print("\nData types:")
print(df_skills.dtypes)
print("\nNull counts:")
print(df_skills.isnull().sum())
print("\nUnique job_ids:", df_skills["job_id"].nunique())
print("Unique skills:", df_skills["skill"].nunique())
print("\nAll unique skills:")
print(sorted(df_skills["skill"].unique()))
print("\nSkill categories:", sorted(df_skills["skill_category"].unique()))
print("Skill levels:", sorted(df_skills["skill_level"].unique()))
print("\nSkill value counts:")
print(df_skills["skill"].value_counts())
print("\nFirst 5 rows:")
df_skills.head()

**`skills_demand.csv` — Schema Documentation**

| Column | Dtype | Nulls | Description |
|--------|-------|-------|-------------|
| `job_id` | str | 0 | Job identifier (links to `ai_jobs.csv`, but may have E-prefix offset) |
| `skill` | str | 0 | Required technical skill (11 unique: Python, SQL, R, AWS, Azure, GCP, TensorFlow, PyTorch, Scikit-learn, NLP, Computer Vision) |
| `skill_category` | str | 0 | Category: Programming, ML, or Cloud |
| `skill_level` | str | 0 | Proficiency: Basic, Intermediate, or Advanced |

> **Key insight:** Each job has ~4–5 skill rows (224,605 rows / 50,000 unique jobs).
> This is already in "long/exploded" format — no need to explode during cleaning.

### 1.2.3 — Inspect Supporting Files

In [ ]:
# country_ai_trends.csv
df_trends = pd.read_csv("data/raw/country_ai_trends.csv")
print("=" * 60)
print("country_ai_trends.csv")
print("=" * 60)
print(f"Shape: {df_trends.shape}")
print(f"Columns: {df_trends.columns.tolist()}")
print(df_trends.head())

print()

# job_title_mapping.csv
df_mapping = pd.read_csv("data/raw/job_title_mapping.csv")
print("=" * 60)
print("job_title_mapping.csv")
print("=" * 60)
print(f"Shape: {df_mapping.shape}")
print(df_mapping)

print()

# data_dictionary.csv
df_dict = pd.read_csv("data/raw/data_dictionary.csv")
print("=" * 60)
print("data_dictionary.csv")
print("=" * 60)
print(df_dict.to_string())

---
## 1.3 — Secondary Dataset Acquisition

**Dataset:** 1.3M LinkedIn Jobs & Skills 2024 (Kaggle: `asaniczka/1-3m-linkedin-jobs-and-skills-2024`)

This dataset contains **3 files**:
| File | Size | Description |
|------|------|-------------|
| `job_skills.csv` | ~673 MB | Job link → comma-separated skills (1.3M rows) |
| `linkedin_job_postings.csv` | ~415 MB | Job metadata: title, company, location, level, type |
| `job_summary.csv` | ~5.1 GB | Full job summaries/descriptions |

> ⚠️ **Do NOT load the full files into memory.** Use `nrows=10000` for initial inspection.

In [ ]:
# Download secondary dataset via kagglehub
import kagglehub
from pathlib import Path

secondary_path = kagglehub.dataset_download("asaniczka/1-3m-linkedin-jobs-and-skills-2024")
print("Path to dataset files:", secondary_path)
print("\nFiles in download:")
for p in sorted(Path(secondary_path).rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p.name:30s}  {size_mb:>10,.1f} MB")

In [ ]:
# Copy all secondary files to data/raw/
import shutil

raw_dir = Path("data/raw")
for f in Path(secondary_path).iterdir():
    if f.is_file():
        dest = raw_dir / f.name
        shutil.copy2(str(f), str(dest))
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"  Copied: {f.name} → data/raw/ ({size_mb:,.1f} MB)")

print("\n✅ Secondary dataset files saved to data/raw/")

### 1.3.1 — Inspect `job_skills.csv` (LinkedIn Skills — 10k sample)

In [ ]:
df_li_skills = pd.read_csv("data/raw/job_skills.csv", nrows=10000)

print("Shape (10k sample):", df_li_skills.shape)
print("\nColumns:", df_li_skills.columns.tolist())
print("\nData types:")
print(df_li_skills.dtypes)
print("\nNull counts:")
print(df_li_skills.isnull().sum())
print("\nFirst 5 rows:")
df_li_skills.head()

### 1.3.2 — Inspect `linkedin_job_postings.csv` (LinkedIn Metadata — 10k sample)

In [ ]:
df_li_posts = pd.read_csv("data/raw/linkedin_job_postings.csv", nrows=10000)

print("Shape (10k sample):", df_li_posts.shape)
print("\nColumns:", df_li_posts.columns.tolist())
print("\nData types:")
print(df_li_posts.dtypes)
print("\nNull counts:")
print(df_li_posts.isnull().sum())
print("\nFirst 5 rows:")
df_li_posts.head()

### 1.3.3 — Column Overlap Analysis: Primary vs. Secondary

In [ ]:
primary_cols = set(pd.read_csv("data/raw/ai_jobs.csv", nrows=1).columns)
secondary_cols = set(pd.read_csv("data/raw/linkedin_job_postings.csv", nrows=1).columns)

overlap = primary_cols & secondary_cols
only_primary = primary_cols - secondary_cols
only_secondary = secondary_cols - primary_cols

print("COLUMN OVERLAP ANALYSIS")
print("=" * 60)
print(f"Primary columns ({len(primary_cols)}):   {sorted(primary_cols)}")
print(f"Secondary columns ({len(secondary_cols)}): {sorted(secondary_cols)}")
print(f"\n✅ Overlapping ({len(overlap)}):     {sorted(overlap)}")
print(f"⬅️  Only in primary ({len(only_primary)}): {sorted(only_primary)}")
print(f"➡️  Only in secondary ({len(only_secondary)}): {sorted(only_secondary)}")

print("\n" + "=" * 60)
print("SCHEMA DIFFERENCES SUMMARY")
print("=" * 60)
print("""
1. Only 'job_title' overlaps between primary (ai_jobs) and secondary (linkedin_job_postings).
2. Primary uses 'job_id' for join keys; Secondary uses 'job_link' (URL).
3. Primary has structured fields: experience_level, salary, posted_year.
   Secondary has: job_level, job_type, first_seen (date), company.
4. Skills are in separate files for both:
   - Primary: skills_demand.csv (structured: skill + category + level)
   - Secondary: job_skills.csv (comma-separated text string)
5. The secondary dataset is from 2024 only; primary spans 2020–2026.
""")

### 1.3.4 — Extract and Save LinkedIn 10k-Row Sample

In [ ]:
# Load 10k rows from both LinkedIn files
df_li_skills_sample = pd.read_csv("data/raw/job_skills.csv", nrows=10000)
df_li_posts_sample = pd.read_csv("data/raw/linkedin_job_postings.csv", nrows=10000)

# Merge skills with postings on job_link to create a combined sample
df_li_merged = df_li_posts_sample.merge(df_li_skills_sample, on="job_link", how="inner")
print(f"LinkedIn merged sample: {df_li_merged.shape[0]} rows (inner join on job_link)")
print(f"Columns: {df_li_merged.columns.tolist()}")

# Save the sample
df_li_skills_sample.to_csv("data/raw/linkedin_skills_sample_10k.csv", index=False)
print(f"\n✅ Saved: data/raw/linkedin_skills_sample_10k.csv ({len(df_li_skills_sample)} rows)")

df_li_merged.to_csv("data/raw/linkedin_merged_sample.csv", index=False)
print(f"✅ Saved: data/raw/linkedin_merged_sample.csv ({len(df_li_merged)} rows)")

df_li_merged.head()

---
## Summary — All Raw Files in `data/raw/`

### Primary Dataset (Global AI & DS Job Market 2020–2026)
| File | Rows | Cols | Description |
|------|------|------|-------------|
| `ai_jobs.csv` | 50,000 | 14 | Job metadata (title, location, salary, year) |
| `skills_demand.csv` | 224,605 | 4 | Skill per job (already in long format) |
| `country_ai_trends.csv` | ~50 | 6 | Country-level annual trends |
| `job_title_mapping.csv` | 6 | 3 | Title → standardized title mapping |
| `data_dictionary.csv` | 12 | 3 | Column documentation |

### Secondary Dataset (1.3M LinkedIn Jobs & Skills 2024)
| File | Rows | Cols | Description |
|------|------|------|-------------|
| `job_skills.csv` | ~1.3M | 2 | Job link + comma-separated skills |
| `linkedin_job_postings.csv` | ~1.3M | 14 | Job metadata (title, company, location) |
| `job_summary.csv` | ~1.3M | ~3 | Full job descriptions |
| `linkedin_skills_sample_10k.csv` | 10,000 | 2 | ⬆️ 10k sample of job_skills.csv |
| `linkedin_merged_sample.csv` | varies | 15 | ⬆️ Merged sample (postings + skills) |

---
## Notes for Phase 2 (Data Cleaning)

Key findings that affect the cleaning strategy:

1. **Skills are in a separate file** (`skills_demand.csv`), not embedded in `ai_jobs.csv`.
   - The implementation plan assumed a `skills_required` column inside a single CSV — that does not exist.
   - Instead, `skills_demand.csv` is already in **long/exploded format** (one row per skill per job).
   - The plan's skill extraction logic can be simplified: we just need to JOIN, not parse + explode.

2. **Date column** — The primary dataset has `posted_year` (integer), not `date_posted` (datetime).
   - Monthly/quarterly trends will need to use the secondary dataset's `first_seen` column or be done at annual granularity.

3. **Job ID mismatch** — `skills_demand.job_id` values sometimes have an "E" prefix offset vs. `ai_jobs.job_id`.
   - Cleaning will need a fuzzy join or prefix-strip strategy.

4. **Only 11 unique skills** in the primary dataset (Python, SQL, R, AWS, Azure, GCP, TensorFlow, PyTorch, Scikit-learn, NLP, Computer Vision).
   - GenAI skills (LLM, RAG, Prompt Engineering) are **not present** in the primary dataset.
   - These will need to be extracted from the LinkedIn `job_skills` free-text column.

5. **LinkedIn schema** has `job_level` (not `experience_level`) and `first_seen` (not `posted_year`).